<a href="https://colab.research.google.com/github/AbdAlRahman-Odeh-99/Two_Phases_Simulation/blob/main/gmm_2class_bandit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import os
import sys
import pandas as pd
import argparse
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
import scipy.optimize as opt
from scipy.stats import norm
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment

def generate_data(rng,nsamples=1000,nclasses=2,nviews=10,seed=42):
    mm = rng.random(size=(nclasses,nviews)) * 2 # symmetric
    # shared variances
    # FIXME: currently simplify to unit variances, only means differ
    stds = 1.0 # simplification
    #mm = np.concatenate([-mm, mm],axis=0) #
    data = make_blobs(nsamples,n_features=nviews,centers=mm,cluster_std=stds,random_state=seed)
    return data[0], mm, data[1]

def optimal_static(means,stds,costs,budget,num_rounds):
    # given the means (symmetric) and costs
    # compute the error probability and let the accuracy (1-err) be the expected reward
    # it is the qfunction of mean squares divided by the standard variances
    nviews = len(costs)
    # the statistics are
    # mean differences
    mean_diff_sq = np.square( 0.5*(means[0,:] - means[1,:])/stds )
    #stats = np.square(means/stds)
    lp_table = np.zeros((2**nviews-1,2)) # (expected reward, costs)
    for comb in range(1,2**nviews):
        sel_idx = np.array([int(bit) for bit in f"{comb:0{nviews}b}"]).astype(bool)
        snrs = np.sum(mean_diff_sq[sel_idx])
        err_rate_calc = norm.sf(np.sqrt(snrs), loc=0,scale=1) # survival function

        cost_sum = np.sum(costs[sel_idx])
        # offset the index by one as range starts from 1 instead of 0
        lp_table[comb-1,:] = np.array([1-err_rate_calc,cost_sum])

    # use linear program to solve the optimal constrained training
    budget_ratio = budget / num_rounds
    r_expect = np.expand_dims(lp_table[:,0],axis=0)
    r_cost = np.expand_dims(lp_table[:,1],axis=0)
    simplex_sum = np.ones((1,2**nviews-1))
    x_bound = [(0,None) for _ in range(2**nviews-1)]
    res = opt.linprog(-r_expect,A_ub=r_cost,b_ub=budget_ratio,A_eq=simplex_sum, b_eq=1,bounds=x_bound)
    rt_msg = res.fun
    opt_pi = res.x # this is the optimal static distribution
    # provide the planning output as the selection distribution for the subsets
    opt_reward = num_rounds * r_expect @ opt_pi
    opt_costs = num_rounds * r_cost @ opt_pi

    print(f"Rounds:{num_rounds}, Budget:{budget}, Optimal Reward: {opt_reward.item():.4e}, Optimal_cost: {opt_costs.item():.4e}")
    return {"solver_message":rt_msg,"reward":opt_reward.item(),"cost":opt_costs.item(),"solution":opt_pi}

def get_subset(est_means_sq,costs,lambda_dual,spending_ratio):
    best_reward = -np.inf
    best_subset = None
    nviews = len(costs)
    #est_means_sq = np.square(0.5 * (est_means[0,:] - est_means[1,:])/1.0) # assume std =1.0
    for v in range(1, 2**nviews):
        subset = np.array([int(bit) for bit in f"{v:0{nviews}b}"]).astype(bool)
        tmp_cost = np.sum(costs[subset])
        snrs = est_means_sq[subset]
        tmp_err_rate = norm.sf(np.sqrt(np.sum(snrs)), loc=0, scale=1)
        tmp_reward = 1-tmp_err_rate - lambda_dual * (tmp_cost - spending_ratio)
        if tmp_reward > best_reward:
            best_reward = tmp_reward
            best_subset = subset
    # greedily choose the maximum
    return best_subset


def label_matching(y_true, y_pred):
    assert(len(y_pred) == len(y_true))
    # 1. Compute confusion matrix
    D = confusion_matrix(y_true, y_pred)
    # 2. Solve the linear sum assignment problem
    # We negate the matrix because the algorithm minimizes cost, and we want to maximize matches
    row_ind, col_ind = linear_sum_assignment(D.max() - D)
    # 3. Create a mapping from predicted to true labels
    mapping = {col: row for row, col in zip(row_ind, col_ind)}
    # 4. Apply the mapping to the predictions
    y_pred_mapped = np.array([mapping[label] for label in y_pred])
    return y_pred_mapped, mapping

def simulation(num_views,costs,num_rounds,budget,x_data,y_labels):
    # initialize the estimators
    # for simplicity, assume estimating means only
    est_means = np.array(np.concatenate([np.zeros((1,num_views)),-np.ones((1,num_views))],axis=0))
    est_counts = np.ones((2,num_views)) # add a dummy count

    remain_budget = budget
    spending_ratio = budget / num_rounds
    acc_reward = 0
    omd_lambda = 0
    lambda_max = 10 # change this when needed
    step_size = 1.0
    noise_var = 1.0
    # optimistic
    xi_fixed = 2.0

    # recording
    record_acc = np.zeros((num_rounds))
    record_pred = np.zeros((num_rounds))
    for nr in range(num_rounds):
        # add the optimistic means
        mean_diff = est_means[0,:] - est_means[1,:]
        optimistic_means_sq = 0.25 * np.square(mean_diff/noise_var) + np.sqrt(xi_fixed * np.log(nr+1)/est_counts.sum(axis=0))
        subset = get_subset(optimistic_means_sq,costs,omd_lambda,spending_ratio)
        if remain_budget <=0:
            subset = [0] # only the free view is possible
        y_true = y_labels[nr] # NOTE: only for evaluation, not involved in training
        x_observe = x_data[nr,subset]

        # prediction
        # Use all features for prediction, with non-observed ones being 0
        boundary = 0.5 * (est_means[0,subset] + est_means[1,subset])
        mean_diff = est_means[0,subset] - est_means[1,subset]
        y_pred = 0 if np.sum( (boundary - x_observe) * mean_diff) >= 0 else 1
        record_pred[nr] = y_pred
        #acc_reward += int(y_pred == y_true)

        # algorithm update
        instant_cost = np.sum(costs[subset])
        remain_budget = max(0, remain_budget - instant_cost)
        # parameter update
        scaler = np.squeeze(1 - 1/(est_counts[y_true,subset]+1))
        est_means[y_true,subset] = scaler * est_means[y_true,subset] + (1-scaler) * x_observe
        est_counts[y_true,subset] += 1


        raw_lambda = omd_lambda + step_size * (instant_cost - spending_ratio)
        omd_lambda = max(0, min(lambda_max, raw_lambda))
        # recording
        #record_acc[nr] = acc_reward/ (nr+1)

    # do a label matching here
    y_pred_mapped, _ = label_matching(y_labels,record_pred)
    acc_reward = np.sum(y_pred_mapped == y_labels)
    record_acc = np.cumsum(y_pred_mapped == y_labels) / np.arange(1,num_rounds+1)


    avg_acc = acc_reward / num_rounds
    return {"reward": acc_reward, "avg_acc":avg_acc, "record_acc":record_acc, "spending":budget - remain_budget}

# Multi-View Classification Simulation

This notebook provides a framework for simulating multi-view classification scenarios, including data generation, an optimal static baseline, and a dynamic online learning algorithm.

First, let's define the simulation parameters.

In [2]:
# Simulation Parameters
SEED = 42                 # Random seed for simulation
NUM_VIEWS = 5             # Number of different views (features)
NUM_ROUNDS = 1000         # Number of simulation rounds (time steps)
NUM_TRIALS = 20           # Number of independent trials for averaging results
NUM_SAMPLES = 1000        # Number of samples for data generation in each trial
BUDGET_RATIO = 0.7        # Budget ratio relative to total possible cost
OUTPUT_FILE = "output_gmm2_su.csv" # Output CSV file name

## Data Generation

The `generate_data` function creates synthetic data using `make_blobs` from `sklearn.datasets`. It generates features for multiple views and corresponding class labels. The `means` parameter controls the centers of the blobs for each class and view, and `stds` simplifies to unit variances.

## Optimal Static Strategy

The `optimal_static` function calculates the best possible reward (accuracy) and corresponding cost under a given budget by pre-selecting an optimal set of views. It formulates this as a linear programming problem, considering all possible subsets of views and their associated error rates and costs. The `norm.sf` function is used to compute the survival function, which gives the error probability.

## Subset Selection Helper

The `get_subset` function is a helper used by the online learning algorithm. Given estimated means (squared signal-to-noise ratios), costs, a dual variable `lambda_dual`, and a `spending_ratio`, it greedily selects the subset of views that maximizes a reward function, which balances accuracy improvement against cost, adjusted by the dual variable.

## Label Matching for Evaluation

The `label_matching` function addresses the label permutation problem in clustering and classification evaluation where predicted cluster/class labels might not directly correspond to true labels. It uses the Hungarian algorithm (via `scipy.optimize.linear_sum_assignment`) on the confusion matrix to find the optimal mapping between predicted and true labels, maximizing the accuracy.

## Online Simulation Algorithm

The `simulation` function implements an online learning algorithm. In each round, it estimates the means of the features for each class and uses an optimistic strategy to select a subset of views based on these estimates and current budget constraints. It then updates the estimates based on observed data and adjusts a dual variable (`omd_lambda`) to manage the budget. Finally, it uses label matching to evaluate the overall accuracy.

## Running the Simulation

Now, let's execute the simulation across multiple trials using the defined parameters. For each trial, new data is generated, optimal static results are calculated, and the online simulation is run. The results (reward, spending, average accuracy, and optimal reward) are collected and saved to a CSV file.

In [3]:
rng = np.random.default_rng(seed=SEED)
result_pd = {"trial":[], "reward":[], "spending":[], "avg_acc":[],"optimal_reward":[]}

for trial in range(NUM_TRIALS):
    budget = NUM_ROUNDS * BUDGET_RATIO
    costs = rng.random(size=(NUM_VIEWS,))
    # Create a free view (first view has zero cost)
    costs[0] = 0
    # Normalize costs to sum to 1
    costs = costs/np.sum(costs)

    # Generate data for the current trial
    t_data, t_means, t_labels = generate_data(rng, NUM_SAMPLES, 2, NUM_VIEWS, SEED)

    # Calculate optimal static baseline
    opt_dict = optimal_static(t_means, 1.0, costs, budget, NUM_ROUNDS)

    # Run the online simulation
    sim_dict = simulation(NUM_VIEWS, costs, NUM_ROUNDS, budget, t_data, t_labels)

    # Accumulate results
    result_pd['trial'].append(trial)
    result_pd['reward'].append(sim_dict['reward'])
    result_pd['spending'].append(sim_dict['spending'])
    result_pd['avg_acc'].append(sim_dict['avg_acc'])
    result_pd['optimal_reward'].append(opt_dict['reward'])

# Convert results to DataFrame and save to CSV
res_df = pd.DataFrame.from_dict(result_pd)
res_df.to_csv(OUTPUT_FILE, index=False)

print(f"Simulation completed. Results saved to {OUTPUT_FILE}")
print(res_df.head())

Rounds:1000, Budget:700.0, Optimal Reward: 8.2637e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 9.0613e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 7.8387e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 8.5217e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 7.7821e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 8.0774e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 8.3364e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 8.0593e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 7.9231e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 7.4974e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 8.6181e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Reward: 8.3463e+02, Optimal_cost: 7.0000e+02
Rounds:1000, Budget:700.0, Optimal Rewar